In [1]:
if 'agg' in globals():
    out_dir = root / 'data' / 'processed' / 'municipio_goias'
    out_dir.mkdir(parents=True, exist_ok=True)
    agg.to_parquet(out_dir / 'municipio_agg_goias.parquet', index=False)
    print('Arquivo salvo em:', out_dir / 'municipio_agg_goias.parquet')
else:
    print('Sem dados para exportar.')

print('\nMetodologia: a correlação entre perfil do eleitorado e voto em nível municipal não indica comportamento individual; isso é falácia ecológica e deve ser destacado na narrativa final.')

Sem dados para exportar.

Metodologia: a correlação entre perfil do eleitorado e voto em nível municipal não indica comportamento individual; isso é falácia ecológica e deve ser destacado na narrativa final.


## 6. Salvar resultados e relatórios

A última etapa prepara os dados processados e resumos para uso em análise posterior, mapas e apresentação para GitHub/LinkedIn.

In [2]:
if 'agg' in globals():
    plt.figure(figsize=(10, 5))
    plt.hist(agg['total_eleitores'], bins=25, edgecolor='black')
    plt.title('Distribuição do total de eleitores por município')
    plt.xlabel('Eleitores')
    plt.ylabel('Municípios')
    plt.tight_layout()
    plt.show()
else:
    print('Execute a agregação antes de plotar.')

Execute a agregação antes de plotar.


## 5. Criar visualizações iniciais

Gráficos permitem identificar padrões de distribuição, diferenças entre municípios e proporções por segmentos do eleitorado antes da modelagem.

A seguir, o painel foca em composição do eleitorado, território e estrutura demográfica municipal, sem interpretar isso como voto por candidato. Essa é a leitura correta com a base disponível.

In [3]:
if 'cleaned' in globals():
    agg = cleaned.groupby('nm_municipio', as_index=False).agg(
        total_eleitores=('qt_eleitores', 'sum'),
        total_secoes=('nr_secao', 'nunique'),
        total_zonas=('nr_zona', 'nunique')
    )

    genero = (
        cleaned.groupby(['nm_municipio', 'ds_genero'], as_index=False)['qt_eleitores']
        .sum()
        .pivot(index='nm_municipio', columns='ds_genero', values='qt_eleitores')
        .fillna(0)
        .reset_index()
    )

    if 'FEMININO' in genero.columns and 'MASCULINO' in genero.columns:
        genero['total_genero'] = genero[['FEMININO', 'MASCULINO']].sum(axis=1)
        genero['share_feminino'] = (genero['FEMININO'] / genero['total_genero'] * 100).fillna(0)
        genero['share_masculino'] = (genero['MASCULINO'] / genero['total_genero'] * 100).fillna(0)
        agg = agg.merge(genero[['nm_municipio', 'share_feminino', 'share_masculino']], on='nm_municipio', how='left')

    faixa_jovem = ['16 a 19 anos', '20 a 24 anos', '25 a 29 anos']
    jovens = cleaned[cleaned['ds_faixa_etaria'].isin(faixa_jovem)] if 'ds_faixa_etaria' in cleaned.columns else None
    if jovens is not None:
        jovens = jovens.groupby('nm_municipio', as_index=False)['qt_eleitores'].sum()
        jovens.columns = ['nm_municipio', 'eleitores_jovens_16_29']
        agg = agg.merge(jovens, on='nm_municipio', how='left')
        agg['share_jovens_16_29'] = (agg['eleitores_jovens_16_29'] / agg['total_eleitores'] * 100).fillna(0)

    top = agg.sort_values('total_eleitores', ascending=False).head(15).copy()

    fig, axes = plt.subplots(2, 2, figsize=(18, 12))

    if 'share_masculino' in agg.columns and 'share_feminino' in agg.columns:
        axes[0, 0].scatter(
            agg['share_masculino'],
            agg['share_feminino'],
            s=agg['total_eleitores'] / 15,
            c=agg['total_eleitores'],
            cmap='viridis',
            alpha=0.7
        )
        axes[0, 0].set_title('Composição por sexo dos municípios')
        axes[0, 0].set_xlabel('% eleitores do sexo masculino')
        axes[0, 0].set_ylabel('% eleitores do sexo feminino')
        axes[0, 0].grid(alpha=0.2)

    if 'share_feminino' in agg.columns:
        axes[0, 1].hist(agg['share_feminino'], bins=20, edgecolor='black', color='#4c72b0')
        axes[0, 1].set_title('Distribuição do % de eleitores femininos')
        axes[0, 1].set_xlabel('% feminino')
        axes[0, 1].set_ylabel('Municípios')

    axes[1, 0].bar(top['nm_municipio'], top['total_eleitores'], color='#55a868')
    axes[1, 0].set_title('Top 15 municípios por total de eleitores')
    axes[1, 0].set_xlabel('Município')
    axes[1, 0].set_ylabel('Total de eleitores')
    axes[1, 0].tick_params(axis='x', rotation=45)

    if 'share_jovens_16_29' in agg.columns and 'share_feminino' in agg.columns:
        axes[1, 1].scatter(
            agg['share_feminino'],
            agg['share_jovens_16_29'],
            s=agg['total_eleitores'] / 18,
            c=agg['total_eleitores'],
            cmap='plasma',
            alpha=0.7
        )
        axes[1, 1].set_title('Perfil feminino x juventude municipal')
        axes[1, 1].set_xlabel('% eleitores do sexo feminino')
        axes[1, 1].set_ylabel('% eleitores jovens (16 a 29 anos)')
        axes[1, 1].grid(alpha=0.2)

    fig.tight_layout()
    plt.show()
else:
    print('Base não preparada. Execute as células de limpeza e agregação antes de plotar o painel demográfico.')

Base não preparada. Execute as células de limpeza e agregação antes de plotar o painel demográfico.


## 4. Realizar agregações e métricas

A partir da base por seção, calculamos agregações por município, como total de eleitores, número de zonas e seções e distribuições por faixa etária, gênero e escolaridade.

In [4]:
if 'df' in globals():
    cleaned = df.copy()
    cleaned.columns = [str(c).strip().lower().replace(' ', '_') for c in cleaned.columns]
    cleaned['nm_municipio'] = cleaned['nm_municipio'].astype(str).str.strip().str.upper()
    cleaned['qt_eleitores'] = pd.to_numeric(cleaned.get('qt_eleitores', 0), errors='coerce').fillna(0)
    cleaned['nr_zona'] = pd.to_numeric(cleaned.get('nr_zona', 0), errors='coerce').fillna(0)
    cleaned['nr_secao'] = pd.to_numeric(cleaned.get('nr_secao', 0), errors='coerce').fillna(0)
    print(cleaned.head(2).to_string(index=False))
else:
    print('DataFrame ausente. Carregue a base antes.')

DataFrame ausente. Carregue a base antes.


## 3. Limpar e transformar colunas

Nesta etapa, renomeamos colunas, padronizamos texto, convertemos tipos e removemos ruído para preparar a base para agregações e comparação por município.

In [5]:
if 'df' in globals():
    print('Linhas:', df.shape[0])
    print('Colunas:', df.shape[1])
    print('\nTipos de dados:\n', df.dtypes.head(10))
    print('\nValores ausentes por coluna:\n', df.isna().sum().head(10))
    print('\nResumo rápido:\n', df.describe(include='all').T.head())
else:
    print('Carregue a base antes de rodar esta célula.')

Carregue a base antes de rodar esta célula.


## 2. Explorar a estrutura do conjunto de dados

Aqui o objetivo é inspecionar colunas, tipos, linhas, ausência de dados e estatísticas básicas para validar a qualidade da base antes de aplicar filtros e agregações.

In [6]:
from pathlib import Path

raw_dir = root / 'data' / 'raw'
print('Raw dir exists?', raw_dir.exists())

for path in sorted(raw_dir.rglob('*'))[:20]:
    print(path.name)

idata = root / 'data' / 'processed' / 'validation' / '2026' / 'perfil_eleitor_secao_2026_GO.parquet'
print('Parquet disponível?', idata.exists())
if idata.exists():
    df = pd.read_parquet(idata)
    print(df.shape)
    print(df.head(2).to_string(index=False))

NameError: name 'root' is not defined

## 1. Importar bibliotecas e carregar dados

Importamos as bibliotecas necessárias e listamos os dados disponíveis para iniciar a análise. A base real em 2026 já foi identificada como o arquivo de perfil do eleitorado por seção de Goiás.

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

root = Path('..').resolve()
print('Projeto:', root)
print('Estrutura de dados:', list((root / 'data').iterdir()))

# Análise eleitoral para presidente em Goiás: 2022 e 2026

Este notebook organiza a análise exploratória inicial do comportamento eleitoral em Goiás, com foco em presidente e uma comparação entre os anos de 2022 e 2026.

Objetivos:
- identificar padrões por município
- comparar anos e perfis socioeconômicos
- preparar base para mapas e clustering
- documentar limitações metodológicas